# Deep Image Prior: single-image super-resolution

Cloud-ready baseline for the assignment. Run each model/optimization variation as a separate W&B run and keep the seed, image, and scale fixed when comparing results.

In [ ]:
# Colab/Kaggle setup (run once per kernel)
import sys
from pathlib import Path
import subprocess

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((path for path in candidates if (path / "requirements.txt").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a checkout containing requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt
import torch
from skimage import data

from dip_sr import ExperimentConfig, make_image_pair, psnr, run_experiment, tensor_to_image

# Replace this with a path or your own numpy HWC image for the final study.
image = data.astronaut()
scale = 4
pair = make_image_pair(image, scale=scale)
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"HR: {tuple(pair.hr.shape[-2:])}, LR: {tuple(pair.lr.shape[-2:])}")

In [ ]:
config = ExperimentConfig(
    scale=scale,
    iterations=2000,
    channels=64,
    seed=42,
    experiment_name="dip-baseline",
    wandb_project=None,  # Set to e.g. "dip-super-resolution" after wandb.login().
    wandb_mode="online",
)
result = run_experiment(pair, config)
print(result.history[-1])

In [ ]:
bicubic = torch.nn.functional.interpolate(pair.lr, size=pair.hr.shape[-2:], mode="bicubic", align_corners=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title("HR reference")
axes[1].imshow(tensor_to_image(bicubic)); axes[1].set_title(f"Bicubic ({psnr(bicubic, pair.hr):.2f} dB)")
axes[2].imshow(tensor_to_image(result.output)); axes[2].set_title(f"DIP ({psnr(result.output, pair.hr):.2f} dB)")
for axis in axes: axis.axis("off")
plt.tight_layout()
plt.show()

## Next variation

Copy the baseline config/run cell, change one factor only (architecture, noise, regularization, or optimization), and give it a new `experiment_name`. Add a row to the visual comparison and use the W&B dashboard for the loss/PSNR curves and run comparison.